In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallTimeseries"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"#; spinup_hours="-16"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#GETTING PRECIP DATA

In [ ]:
# #Loading Radar Mask #decided not to use here
# RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData_NSSL) 

In [ ]:
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T

In [ ]:
def GetMean(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

In [ ]:
# #DATA LOADING FOR NCEP/EMC LEVEL IV DATA (*OLD*)

# def ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory, region='conus'):
#     yearmonth = yearmonthdayhour[0:6]
#     inputPath = os.path.join(inputDirectory,yearmonth,f"st4_{region}.{yearmonthdayhour}.01h.grb2")
#     precipData = xr.open_dataset(inputPath, engine="cfgrib")['tp']
#     #Note: data in kg/m^2 = mm since rho = m/V = m/A/h where m/A = 1 ==> h = 1e-3 m = 1 mm
#     return precipData

# def GetAccumulatedPrecipData_LevelIV(ModelData): 
#     #getting inputDirectory
#     inputDirectory = os.path.join(DirectoryManager.dataDirectory,
#                                   f"Observation_Data/{ModelData.region}/StageIV_PrecipData")
#     print(f"reading from {inputDirectory}")

#     #getting yearmonthdayhour list
#     dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
#     start = dt_list[0]
#     target = start + timedelta(hours=12)
    
#     # Find the entry closest to +12 hours
#     closest = min(dt_list, key=lambda x: abs(x - target))
    
#     closest_idx = dt_list.index(closest)
#     times = ModelData.timeStrings[closest_idx:]
#     yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
#                                 for t in times})
    
#     for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
#         if count == 0:
#             precipData_T = ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory)
#         else:
#             precipData_T += ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory)

#     precipData_T = precipData_T.assign_coords(longitude = precipData_T.longitude - 360)
#     return precipData_T

# precipData_T = GetAccumulatedPrecipData_LevelIV(ModelData_NSSL)
# precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion_Curvilinear(precipData_T, ModelData_NSSL)

In [ ]:
def GetAccumulatedPrecipData_Model(ModelData): 
    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start #+ timedelta(hours=13) #only using for rainfall histogram
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]

    precipData_model_T_Subset_Mean = []; timeList = []
    for t in tqdm(times):
        data = ModelData.GetDataTimestep(t,printout=False)
        precipData_model_T = data['rainnc']+data['rainc']
    

        # --- subset the region ---
        precipData_model_T_Subset = DataSubsetting_Class.SubsetDataRegion(
            precipData_model_T, ModelData
        )

        # precipData_model_T_Subset=precipData_model_T_Subset.where(RadarDataMask == True) #decided not to use here
        
        # --- mean of the subset domain ---
        meanValue = GetMean(precipData_model_T_Subset).item()
        precipData_model_T_Subset_Mean.append(meanValue)
        timeList.append(t)

    timeList = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in timeList]
    return np.array(precipData_model_T_Subset_Mean),np.array(timeList)

def LoadOrRun_GetAccumulatedPrecipData_Model(
    ModelData,
    overwrite=False
):
    """
    Load or compute model domain-mean accumulated precipitation time series.
    Always returns Python datetime.datetime objects for time.
    """

    os.makedirs(outputDirectory, exist_ok=True)

    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs_"
        f"{ModelData.mpType}_"
        f"Model_Precipitation_Timeseries.nc"
    )
    filePath = os.path.join(outputDirectory, fileName)

    # --------------------------------------------------
    # LOAD
    # --------------------------------------------------
    if os.path.exists(filePath) and not overwrite:
        print("Loading model precip time series from disk:")
        print(f"  {filePath}")

        with xr.open_dataset(filePath) as ds:
            precipMean = ds["precip_mean"].values
            time_np = ds["time"].values

        # Convert numpy.datetime64 → datetime.datetime
        timeArray = np.array(
            [t.astype("datetime64[ms]").astype(datetime) for t in time_np],
            dtype=object
        )

        return precipMean, timeArray

    # --------------------------------------------------
    # RUN
    # --------------------------------------------------
    print("Computing model accumulated precip time series")
    precipMean, timeArray = GetAccumulatedPrecipData_Model(ModelData)
    # timeArray is already Python datetime here

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------
    ds = xr.Dataset(
        data_vars={
            "precip_mean": ("time", precipMean),
        },
        coords={
            # NetCDF requires datetime64
            "time": np.array(timeArray, dtype="datetime64[ms]"),
        }
    )

    ds["precip_mean"].attrs["description"] = (
        "Domain-mean accumulated model precipitation (rainnc + rainc)"
    )
    ds.attrs["region"] = ModelData.region
    ds.attrs["case"] = ModelData.case
    ds.attrs["spinup_hours"] = ModelData.spinup_hours
    ds.attrs["microphysics"] = ModelData.mpType

    ds.to_netcdf(filePath)
    print("Saved model precip time series to disk:")
    print(f"  {filePath}")

    return precipMean, timeArray


In [ ]:
#DATA LOADING FOR MRMS QPE DATA

def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates
    
def GetMRMS_QPE_DataDirectory(ModelData, product="MultiSensor_QPE_01H_Pass2_00.00"):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/{ModelData.region}/MRMS_RadarData",
             f"{simulationDates2[0]}_{simulationDates2[-1]}",product)
    return inputDirectory
    
def ReadPrecipData_MRMS_QPE(ModelData, yearmonthdayhour, inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]

    MRMS_region = "CONUS" if ModelData.region=="TRACER" else "HAWAII"
    inputPath = os.path.join(inputDirectory,f"MRMSQPE_{MRMS_region}_{ModelData.region}_{yearmonthday}-{hour}0000.nc")
    try:
        precipData = xr.open_dataset(inputPath)["MultiSensor_QPE_01H_Pass2_00.00"].isel(time=0)
    except:
        print(f"{inputPath} does not exist in MRMS data ==> skipping")
        precipData = None
    #Note: data is in mm units
    return precipData

def GetAccumulatedPrecipData_MRMS_QPE(ModelData): 
    #getting inputDirectory
    inputDirectory = GetMRMS_QPE_DataDirectory(ModelData)
    print(f"reading from {inputDirectory}")

    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start# + timedelta(hours=13) #only using for rainfall histogram
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]
    yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
                                for t in times})

    precipData_T = None
    precipData_T_Subset_Mean = []; timeList = []
    for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
        # print(yearmonthdayhour)
        
        precipData = ReadPrecipData_MRMS_QPE(ModelData,yearmonthdayhour,inputDirectory)
        if precipData is None:
            continue #skips missing hours
        if precipData_T is None:
            precipData_T = precipData #equates at first timestep
        else:
            precipData_T += precipData
            
        # --- make a copy for coordinate fixes ---
        precipData_T_2 = precipData_T.assign_coords(
            longitude = precipData_T.longitude - 360
        ).sortby("latitude")

        # --- subset the region ---
        precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion(
            precipData_T_2, ModelData
        )

        # # --- applying radarDataMask --- #decided not to use here
        # RadarDataMask_interp = RadarDataMask.astype(float).interp(latitude = precipData_T_Subset.latitude,
        #                                         longitude = precipData_T_Subset.longitude,
        #                                         method='nearest')
        # RadarDataMask_interp = RadarDataMask_interp ==1
        # precipData_T_Subset=precipData_T_Subset.where(RadarDataMask_interp == True)

        # --- mean of the subset domain ---
        meanValue = GetMean(precipData_T_Subset).item()
        precipData_T_Subset_Mean.append(meanValue)
        timeList.append(yearmonthdayhour)

    timeList = [datetime.strptime(t, "%Y%m%d%H") for t in timeList]
    return np.array(precipData_T_Subset_Mean),np.array(timeList)

def LoadOrRun_GetAccumulatedPrecipData_MRMS_QPE(
    ModelData,
    overwrite=False
):
    """
    Load or compute MRMS QPE domain-mean accumulated precipitation time series.
    Always returns Python datetime.datetime objects for time.
    """

    os.makedirs(outputDirectory, exist_ok=True)

    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs_"
        f"MRMS_QPE_Precipitation_Timeseries.nc"
    )
    filePath = os.path.join(outputDirectory, fileName)

    # --------------------------------------------------
    # LOAD
    # --------------------------------------------------
    if os.path.exists(filePath) and not overwrite:
        print("Loading MRMS QPE precip time series from disk:")
        print(f"  {filePath}")

        with xr.open_dataset(filePath) as ds:
            precipMean = ds["precip_mean"].values
            time_np = ds["time"].values

        # 🔑 Convert numpy.datetime64 → datetime.datetime
        timeArray = np.array(
            [t.astype("datetime64[ms]").astype(datetime) for t in time_np],
            dtype=object
        )

        return precipMean, timeArray

    # --------------------------------------------------
    # RUN
    # --------------------------------------------------
    print("Computing MRMS QPE accumulated precip time series")
    precipMean, timeArray = GetAccumulatedPrecipData_MRMS_QPE(ModelData)
    # timeArray is already Python datetime here

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------
    ds = xr.Dataset(
        data_vars={
            "precip_mean": ("time", precipMean),
        },
        coords={
            # NetCDF-safe datetime
            "time": np.array(timeArray, dtype="datetime64[ms]"),
        }
    )

    ds["precip_mean"].attrs["description"] = (
        "Domain-mean accumulated MRMS QPE precipitation"
    )
    ds.attrs["dataset"] = "MRMS_QPE"
    ds.attrs["region"] = ModelData.region
    ds.attrs["case"] = ModelData.case
    ds.attrs["spinup_hours"] = ModelData.spinup_hours

    ds.to_netcdf(filePath)
    print("Saved MRMS QPE precip time series to disk:")
    print(f"  {filePath}")

    return precipMean, timeArray


In [ ]:
[precipData_NSSL, timeList_NSSL] = LoadOrRun_GetAccumulatedPrecipData_Model(ModelData_NSSL)
[precipData_TEMPO, timeList_TEMPO] = LoadOrRun_GetAccumulatedPrecipData_Model(ModelData_TEMPO)

In [ ]:
[precipData_MRMS,timeList_MRMS] = LoadOrRun_GetAccumulatedPrecipData_MRMS_QPE(ModelData_NSSL)

In [ ]:
def GetdtMinutes(time):
    dtMinutes = [(time[i+1] - time[i]).total_seconds() / 60 
                  for i in range(len(time)-1)]
    return dtMinutes[0]

def CalculateRainRate(R,time):
    # R = outputDictionary['rainnc+rainc']
    # dtMinutes = 15
    dtMinutes = GetdtMinutes(time)
    dtHours = dtMinutes / 60
    rainRate = (R[1:] - R[:-1]) / dtHours

    # rainRateTime = time[1:]
    rainRate = np.insert(rainRate, 0, np.nan)
    rainRateTime = time
    return rainRate, rainRateTime

In [ ]:
rainRate_NSSL, rainRateTime_NSSL = CalculateRainRate(precipData_NSSL,timeList_NSSL)
rainRate_TEMPO, rainRateTime_TEMPO = CalculateRainRate(precipData_TEMPO,timeList_TEMPO)
rainRate_MRMS, rainRateTime_MRMS = CalculateRainRate(precipData_MRMS,timeList_MRMS)

In [ ]:
###################
#PLOTTING FUNCTIONS
fontSettings = {
    "tickFont": 16,
    "labelFont": 18,
    "legendFont": 14,
    "titleFont": 22,
}

In [ ]:
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    import numpy as np
    from matplotlib.dates import date2num

    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())

def lineplot(axis, time, output, varName, units, color, label):
    axis.plot(time, output.squeeze(), color=color, label=label)
    axis.set_ylabel(f"{varName} " + fr"$({units})$", fontsize=fontSettings["labelFont"])
    axis.set_xlabel("Time", fontsize=fontSettings["labelFont"])
    axis.grid(True)
    SetXLimitsDatetime(axis, time)
    plt.setp(axis.get_xticklabels(), rotation=45, ha="right")

    # ---- Tick font size control ----
    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

In [ ]:
def MakeCombinedPlot_1():
    
    #Accumulated Rainfall Timeseries
    
    fig = plt.figure(figsize=(16,6))
    gs  = fig.add_gridspec(1, 2, wspace=0.15)
    axis1 = fig.add_subplot(gs[0,0])
    
    lineplot(axis1,
             time=timeList_NSSL, output=precipData_NSSL, 
             varName="rainnc+rainc", units="mm", color="blue", label="NSSL")
    lineplot(axis1,
             time=timeList_MRMS, output=precipData_MRMS, 
             varName="rainnc+rainc", units="mm", color="black", label="MRMS QPE")
    lineplot(axis1,
             time=timeList_TEMPO, output=precipData_TEMPO,
             varName="rainnc+rainc", units="mm", color="green", label="TEMPO")
    axis1.legend(loc='upper left')
    
    #RainRate Timeseries
    axis2 = fig.add_subplot(gs[0,1])
    lineplot(axis2,
             time=rainRateTime_NSSL, output=rainRate_NSSL, 
             varName="rainnc+rainc", units="mm/hr", color="blue", label="NSSL")
    lineplot(axis2,
             time=rainRateTime_MRMS, output=rainRate_MRMS, 
             varName="d/dt (rainnc+rainc)", units="mm/hr", color="black", label="MRMS QPE")
    lineplot(axis2,
             time=rainRateTime_TEMPO, output=rainRate_TEMPO, 
             varName="d/dt (rainnc+rainc)", units="mm/hr", color="green", label="TEMPO")
    axis2.legend(loc='upper right')

    #Adding Title
    fig.suptitle(
        f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
        fontsize=20,
        y=0.96,
        fontweight="bold"
    )
    return fig

In [ ]:
def MakeCombinedPlot_2():
    # RainRate Timeseries only
    fig, axis = plt.subplots(figsize=(9,4))

    lineplot(axis,
             time=rainRateTime_NSSL, output=rainRate_NSSL, 
             varName="rainnc+rainc", units="mm/hr", color="blue", label="NSSL")
    lineplot(axis,
             time=rainRateTime_MRMS, output=rainRate_MRMS, 
             varName="d/dt (rainnc+rainc)", units="mm/hr", color="black", label="MRMS QPE")
    lineplot(axis,
             time=rainRateTime_TEMPO, output=rainRate_TEMPO, 
             varName="d/dt (rainnc+rainc)", units="mm/hr", color="green", label="TEMPO")
    axis.legend(loc='upper right',fontsize=fontSettings["legendFont"])

    # Adding title
    axis.set_title(f"{ModelData_NSSL.region} {ModelData_NSSL.case}", fontsize=fontSettings["titleFont"], fontweight="bold")
    fontSettings
    return fig

In [ ]:
def SaveFigure(fig, ModelData1, ModelData2, dpi=300,
               plotType="Accumulated+RainRate"):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RainfallTimeseries_{plotType}.png"
    )

    # # --- Save figure ---
    # FigurePlotting_Class.SaveUniformFigure(fig, outputFilePath)
    fig.savefig(outputFilePath, dpi=dpi, bbox_inches="tight",pad_inches=0.02)
    plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
###################
#PLOTTING

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

In [ ]:
fig =  MakeCombinedPlot_1()
SaveFigure(fig,ModelData_NSSL,ModelData_TEMPO,
           plotType="Accumulated+RainRate")

In [ ]:
fig =  MakeCombinedPlot_2()
SaveFigure(fig,ModelData_NSSL,ModelData_TEMPO,
           plotType="RainRate")
fig

In [ ]:
####################################
#PLOTTING ALL SIMULATIONS

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 900

plotting = False #keep false when job array is running
plotting = True

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
                      plotType="Accumulated+RainRate",
                      extension="png"):
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(outputPlottingDirectory, inputSubDirectory)
    # --- File path ---
    inputFilePath = os.path.join(
        load_dir,
        f"{dataType}_{plotType}.{extension}"
    )
    return inputFilePath

def GetFilePaths(plotType):
    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        filePaths.append(GetFigureFilePath(region,case,spinup_hours,plotType))
    return filePaths

In [ ]:
if plotting:
    sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
    from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:
if plotting:
    plotType = "Accumulated+RainRate"
    filePaths = GetFilePaths(plotType=plotType)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 4),
                                                     wspace=0.01,hspace=0.02,
                                                     dpi=900)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=900, saveDirectory=outputPlottingDirectory,fileName=dataType+f"_{plotType}")

In [ ]:
if plotting:
    plotType = "RainRate"
    filePaths = GetFilePaths(plotType=plotType)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 5),
                                                     wspace=0.01,hspace=0.02,
                                                     dpi=900)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=900, saveDirectory=outputPlottingDirectory,fileName=dataType+f"_{plotType}")